In [1]:
import numpy as np

In [3]:
def init_params():
  W1=np.random.randn(784,10)*0.01
  b1=np.zeros((1,10))
  W2=np.random.randn(10,10)*0.01
  b2=np.zeros((1,10))
  return W1,b1,W2,b2

In [5]:
def relu(Z):
  return np.maximum(0,Z)

In [14]:
def relu_derivative(Z):
  return Z>0

In [9]:
def softmax(Z):
    exp = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

In [10]:
def forward_pass(X,W1,b1,W2,b2):
  Z1=np.dot(X,W1)+b1
  A1=relu(Z1)
  Z2=np.dot(A1,W2)+b2
  A2=softmax(Z2)
  return Z1,A1,Z2,A2

In [11]:
def compute_loss(A2, Y):
    m = Y.shape[0]
    log_likelihood = -np.log(A2[range(m), Y])
    return np.sum(log_likelihood) / m

In [27]:
def backpropagation(X,Y,Z1,A1,A2,W2):
  m=X.shape[0]
  dZ2 = A2
  dZ2[range(m), Y] -= 1
  dZ2 /= m

  dW2 = A1.T @ dZ2
  db2 = np.sum(dZ2, axis=0, keepdims=True)

  dA1 = dZ2 @ W2.T
  dZ1 = dA1 * relu_derivative(Z1)

  dW1 = X.T @ dZ1
  db1 = np.sum(dZ1, axis=0, keepdims=True)

  return dW1, db1, dW2, db2

In [17]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr):
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    return W1, b1, W2, b2

In [24]:
from tqdm.notebook import tqdm

def train(X, Y, epochs=100, lr=0.1):
    W1, b1, W2, b2 = init_params()
    for i in tqdm(range(epochs), desc="Training Progress"):
        Z1, A1, Z2, A2 = forward_pass(X, W1, b1, W2, b2)
        loss = compute_loss(A2, Y)
        dW1, db1, dW2, db2 = backpropagation(X, Y, Z1, A1, A2, W2)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr)
        if i % 10 == 0:
            tqdm.write(f"Epoch {i}, Loss: {loss:.4f}")

    return W1, b1, W2, b2

In [20]:
def predict(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_pass(X, W1, b1, W2, b2)
    return np.argmax(A2, axis=1)

In [28]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data.astype(np.float32) / 255.0
Y = mnist.target.astype(np.int64)


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

W1, b1, W2, b2 = train(X_train, Y_train, epochs=100, lr=0.01)

preds = predict(X_test, W1, b1, W2, b2)
acc = np.mean(preds == Y_test)
print(f"Test Accuracy: {acc*100:.2f}%")

Training Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 0, Loss: 2.3026
Epoch 10, Loss: 2.3025
Epoch 20, Loss: 2.3024
Epoch 30, Loss: 2.3023
Epoch 40, Loss: 2.3022
Epoch 50, Loss: 2.3021
Epoch 60, Loss: 2.3020
Epoch 70, Loss: 2.3019
Epoch 80, Loss: 2.3017
Epoch 90, Loss: 2.3016
Test Accuracy: 15.75%
